# ___Themeda triandra - Power Analysis___
------------------

In [1]:
!python --version

Python 3.13.9


The system cannot find the path specified.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from numba import njit

## ___Data wrangling___
-----------------

In [14]:
# load in the data for soil and climatic properties and root trait data
root_traits = pd.read_csv(r"../data/chapter3/vin_themeda_root_traits.csv", usecols=("Accession", "SRL", "RTD", "Diameter")) # units => m/g, g/cm3, mm
root_traits = root_traits.rename({_: _.lower().strip() for _ in root_traits.columns}, axis=1)
root_traits.loc[:, "accession"] = root_traits.accession.replace(
    {_: _.replace(' ', '_').strip() for _ in root_traits.accession.unique()}
).replace({"Mt_Fox_N_Park_QLD": "Mt_Fox_National_Park_QLD", "Sydney": "Sydney_NSW"}) # replace these two to match with the climate info dataset
root_traits.head()

,accession,srl,rtd,diameter
0,Dalby_QLD,16.292527,0.256367,0.5521
1,Dalby_QLD,35.587354,0.145137,0.4965
2,Dalby_QLD,26.271140,0.332818,0.3816
3,Dalby_QLD,17.405689,0.176030,0.6447
4,Mt_Fox_National_Park_QLD,35.094561,0.265127,0.3699


In [15]:
root_traits.accession.unique()

array(['Dalby_QLD', 'Mt_Fox_National_Park_QLD', 'Panawonica_WA',
       'Rainbow_Valley_NT', 'Sydney_NSW', 'Virginia_Gardens_SA'],
      dtype=object)

In [16]:
climate = pd.read_csv(r"../data/chapter3/site_metereology.csv", skiprows=range(2), usecols=("site", "state", "annual_rainfall_mm", "mean_annual_temp_celsius"))
climate.loc[:, "site"] = (climate.site + '_' + climate.state).str.replace(' ', '_')

In [25]:
pd.merge(left=root_traits, left_on="accession", right=climate.query("site.isin(@root_traits.accession)").reset_index(drop=True), right_on="site", how="left").drop(["site", "state"], axis=1)

,accession,srl,rtd,diameter,annual_rainfall_mm,mean_annual_temp_celsius
0,Dalby_QLD,16.292527,0.256367,0.552100,621.4,27.0
1,Dalby_QLD,35.587354,0.145137,0.496500,621.4,27.0
2,Dalby_QLD,26.271140,0.332818,0.381600,621.4,27.0
3,Dalby_QLD,17.405689,0.176030,0.644700,621.4,27.0
4,Mt_Fox_National_Park_QLD,35.094561,0.265127,0.369900,642.8,29.3
5,Mt_Fox_National_Park_QLD,23.679432,0.214891,0.500300,642.8,29.3
6,Mt_Fox_National_Park_QLD,34.522452,0.297887,0.351900,642.8,29.3
7,Mt_Fox_National_Park_QLD,17.399410,0.194183,0.613900,642.8,29.3
8,Panawonica_WA,21.241229,0.596533,0.317000,404.4,34.7
9,Panawonica_WA,22.604293,0.116710,0.694700,404.4,34.7


In [67]:
# soil properties data
soil = pd.read_csv(r"../data/chapter3/CSBP_soil_analysis_Themeda_and_Sorghum.csv")# , usecols=("Customer Sample ID", "Sample Name 2", ""))
soil = soil.rename({_: _.lower().strip().replace(' ', '_').replace('%', "prcnt").replace('(', '').replace(')', '') for _ in soil.columns}, axis=1) # clean up the column names
soil = soil.drop(["lab_number", "date_received", "customer_sample_id", "sample_name_1", "latitude", "longitude", "depth", "colour", "gravel_percent", "texture"], axis=1) # drop useless columns
soil = soil.query("sample_name_2.isin(('Cobbler Ck SA', 'Dalby Qld', 'Mt Fox Qld', 'Pannawonica WA', 'Rainbow Valley NT', 'Hornsby Heights NSW'))").reset_index(drop=True) # filter the six needed rows
soil.loc[:, "sample_name_2"] = soil.sample_name_2.replace({ # update the site names to match the other datsets
    "Cobbler Ck SA": "Virginia_Gardens_SA", "Dalby Qld": "Dalby_QLD", "Hornsby Heights NSW": "Sydney_NSW", "Mt Fox Qld": "Mt_Fox_National_Park_QLD",
    "Pannawonica WA": "Panawonica_WA", "Rainbow Valley NT": "Rainbow_Valley_NT"
})

In [ ]:
# pH(H2O) vs pH(CaCl2)
# https://agriculture.vic.gov.au/farm-management/soil/understanding-soil-tests-for-pastures
# Soil pH CaCl2 values are usually between 0.5 to 1.1 units lower than pH (water). The pH (water) value readily reflects current soil conditions, but is subject to seasonal variations.
# The CaCl2 test is useful for long term monitoring of pH and is less subject to seasonal variations

In [ ]:
# # https://soilqualityknowledgebase.org.au/resources/soil-phosphorus-testing-colwell-p-and-dgt-p/

# Soil contains phosphorus that can be conceptualised as four pools: solution phosphorus, sorbed phosphorus, mineral phosphorus and organic phosphorus.
# The amount of phosphorus measured by a soil test is dependent on the amount of phosphorus present in each pool and the ability of the soil test method to extract phosphorus from each pool. 
# For the test to be a useful predictor of crop yield responses to fertiliser phosphorus, the test must measure phosphorus that is available to crops for the soil-crop system in question.

# Colewell method for soil P
# In a Colwell phosphorus test, an extract is obtained by adding a soil sample to a sodium bicarbonate solution adjusted to pH 8.5, and agitating for 16 hours. This extract is then acidified before its phosphorus 
# concentration is measured colorimetrically. The Colwell method measures sorbed and solution phosphorus, but does not provide any information on the equilibrium between these two pools, which is determined by the
# phosphors buffering capacity of the soil. The phosphorus buffering index (PBI) is used in combination with Colwell-P to assess the levels of soil P supply to crops and pastures. As the phosphorus buffering index 
# increases, the level of Colwell-P required to provide a sufficient level of phosphorus for crops and pastures increases.

<a src="https://soilqualityknowledgebase.org.au/wp-content/uploads/2023/11/soil-phosphorus-pools-diagram-simple_PNG.png">Image source</a><br><br>

<img src="./soil-phosphorus-pools-diagram-simple_PNG.png" width="400px">

In [70]:
soil

,sample_name_2,ammonium_nitrogen,nitrate_nitrogen,phosphorus_colwell,potassium_colwell,sulfur,organic_carbon,conductivity,ph_level_cacl2,ph_level_h2o,total_nitrogen,total_phosphorus,total_carbon,prcnt_clay,prcnt_course_sand,prcnt_fine_sand,prcnt_sand,prcnt_silt
0,Virginia_Gardens_SA,2,< 1,5,511,10.7,2.28,0.092,6.1,7.0,0.22,258.6,3.07,31.69,18.82,29.01,47.83,20.48
1,Dalby_QLD,6,< 1,18,398,4,2.88,0.069,6.0,6.9,0.27,402.1,3.96,23.11,35.26,26.89,62.15,14.74
2,Sydney_NSW,9,< 1,4,206,4.2,3.26,0.052,4.9,5.9,0.18,229.2,4.04,13.88,51.89,23.51,75.4,10.72
3,Mt_Fox_National_Park_QLD,6,< 1,6,314,3.6,1.71,0.035,5.7,6.7,0.19,157.5,2.66,21.6,33.49,22.22,55.71,22.69
4,Panawonica_WA,2,13,8,385,12.8,1.53,0.232,7.2,8.0,0.14,184.2,2.27,25.64,6,29.82,35.82,38.54
5,Rainbow_Valley_NT,1,< 1,6,125,1.1,0.25,0.061,7.5,8.8,0.02,89.7,0.51,5.8,60.38,31.88,92.26,1.95


In [72]:
# meta info - climate and soil properties
pd.merge(left=climate, left_on="site", right=soil, right_on="sample_name_2").drop(["state", "sample_name_2"], axis=1)

,site,annual_rainfall_mm,mean_annual_temp_celsius,ammonium_nitrogen,nitrate_nitrogen,phosphorus_colwell,potassium_colwell,sulfur,organic_carbon,conductivity,ph_level_cacl2,ph_level_h2o,total_nitrogen,total_phosphorus,total_carbon,prcnt_clay,prcnt_course_sand,prcnt_fine_sand,prcnt_sand,prcnt_silt
0,Rainbow_Valley_NT,189.3,28.9,1,< 1,6,125,1.1,0.25,0.061,7.5,8.8,0.02,89.7,0.51,5.8,60.38,31.88,92.26,1.95
1,Panawonica_WA,404.4,34.7,2,13,8,385,12.8,1.53,0.232,7.2,8.0,0.14,184.2,2.27,25.64,6,29.82,35.82,38.54
2,Dalby_QLD,621.4,27.0,6,< 1,18,398,4,2.88,0.069,6.0,6.9,0.27,402.1,3.96,23.11,35.26,26.89,62.15,14.74
3,Virginia_Gardens_SA,468.7,22.6,2,< 1,5,511,10.7,2.28,0.092,6.1,7.0,0.22,258.6,3.07,31.69,18.82,29.01,47.83,20.48
4,Mt_Fox_National_Park_QLD,642.8,29.3,6,< 1,6,314,3.6,1.71,0.035,5.7,6.7,0.19,157.5,2.66,21.6,33.49,22.22,55.71,22.69
5,Sydney_NSW,1156.9,23.4,9,< 1,4,206,4.2,3.26,0.052,4.9,5.9,0.18,229.2,4.04,13.88,51.89,23.51,75.4,10.72
